# Orchestrator Agent

Coordinates the planner and travel agents via a workflow graph.
Uses MCP to discover agents and routes tasks through them in sequence.

In [1]:
import asyncio
import json
import logging
import os
from collections.abc import AsyncIterable

import anthropic
import uvicorn
import weave
from a2a.types import TaskState
from dotenv import load_dotenv
from google.protobuf import json_format

import prompts
from common import BaseAgent, build_a2a_app
from workflow import Status, WorkflowGraph, WorkflowNode

load_dotenv()
logger = logging.getLogger(__name__)

WANDB_PROJECT = os.getenv('WANDB_WORKSPACE')
weave_client = weave.init(WANDB_PROJECT)
print(f'Weave initialized: {WANDB_PROJECT}')

/Users/paul/.cache/uv/archive-v0/pqxv7ELpHFQOMM0CmHnVk/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(
weave: Logged in as Weights & Biases user: paulbruffett.
weave: View Weave data at https://wandb.ai/pbruffett/a2a-lab/weave


Weave initialized: pbruffett/a2a-lab


In [2]:
class OrchestratorAgent(BaseAgent):
    def __init__(self):
        super().__init__(agent_name='Orchestrator Agent', description='Facilitate inter agent communication', content_types=['text', 'text/plain'])
        self.graph = None
        self.results = []
        self.travel_context = {}
        self.query_history = []
        self.context_id = None

    @weave.op()
    async def _generate_summary(self) -> str:
        client = anthropic.AsyncAnthropic()
        response = await client.messages.create(
            model='claude-sonnet-4-6',
            max_tokens=16000,
            thinking={'type': 'adaptive'},
            messages=[{
                'role': 'user',
                'content': prompts.SUMMARY_COT_INSTRUCTIONS.replace('{travel_data}', str(self.results)),
            }],
        )
        return next((b.text for b in response.content if b.type == 'text'), '')

    @weave.op()
    def _answer_user_question(self, question) -> str:
        try:
            client = anthropic.Anthropic()
            response = client.messages.create(
                model='claude-opus-4-6',
                max_tokens=16000,
                messages=[{
                    'role': 'user',
                    'content': prompts.QA_COT_PROMPT
                        .replace('{TRIP_CONTEXT}', str(self.travel_context))
                        .replace('{CONVERSATION_HISTORY}', str(self.query_history))
                        .replace('{TRIP_QUESTION}', question),
                }],
                output_config={
                    'format': {
                        'type': 'json_schema',
                        'schema': {
                            'type': 'object',
                            'properties': {
                                'can_answer': {'type': 'string', 'enum': ['yes', 'no']},
                                'answer': {'type': 'string'},
                            },
                            'required': ['can_answer', 'answer'],
                            'additionalProperties': False,
                        },
                    },
                },
            )
            return next((b.text for b in response.content if b.type == 'text'), '')
        except Exception:
            return '{"can_answer": "no", "answer": "Cannot answer based on provided context"}'

    def _set_node_attrs(self, node_id, task_id=None, context_id=None, query=None):
        attrs = {}
        if task_id: attrs['task_id'] = task_id
        if context_id: attrs['context_id'] = context_id
        if query: attrs['query'] = query
        self.graph.set_node_attributes(node_id, attrs)

    def _add_graph_node(self, task_id, context_id, query, node_id=None, node_key=None):
        node = WorkflowNode(task=query, node_key=node_key)
        self.graph.add_node(node)
        if node_id:
            self.graph.add_edge(node_id, node.id)
        self._set_node_attrs(node.id, task_id, context_id, query)
        return node

    def _clear_state(self):
        self.graph = None
        self.results.clear()
        self.travel_context.clear()
        self.query_history.clear()

    @weave.op()
    async def stream(self, query, context_id, task_id) -> AsyncIterable[dict[str, any]]:
        logger.info(
            '[orch.stream] REQUEST ctx=%s task=%s query=%r graph_state=%s paused_node=%s',
            context_id, task_id, query,
            (self.graph and self.graph.state),
            (self.graph and self.graph.paused_node_id),
        )
        if not query:
            raise ValueError('Query cannot be empty')
        if self.context_id != context_id:
            self._clear_state()
            self.context_id = context_id

        self.query_history.append(query)
        start_node_id = None

        if not self.graph:
            self.graph = WorkflowGraph()
            planner_node = self._add_graph_node(task_id, context_id, query, node_key='planner')
            start_node_id = planner_node.id
        elif self.graph.state == Status.PAUSED:
            start_node_id = self.graph.paused_node_id
            self._set_node_attrs(start_node_id, query=query)

        while True:
            self._set_node_attrs(start_node_id, task_id=task_id, context_id=context_id)
            should_resume = False

            async for stream_resp, task in self.graph.run_workflow(start_node_id):
                payload_type = stream_resp.WhichOneof('payload')

                if payload_type == 'status_update':
                    evt = stream_resp.status_update
                    context_id = evt.context_id
                    if evt.status.state == TaskState.TASK_STATE_COMPLETED and context_id:
                        continue
                    if evt.status.state == TaskState.TASK_STATE_INPUT_REQUIRED:
                        question = evt.status.message.parts[0].text
                        logger.info('[orch.stream] INPUT_REQUIRED question=%r', question)
                        try:
                            answer = json.loads(self._answer_user_question(question))
                            if answer['can_answer'] == 'yes':
                                query = answer['answer']
                                start_node_id = self.graph.paused_node_id
                                self._set_node_attrs(start_node_id, query=query)
                                should_resume = True
                                logger.info('[orch.stream] auto-answered, resuming with query=%r', query)
                        except Exception:
                            pass

                elif payload_type == 'artifact_update':
                    artifact = stream_resp.artifact_update.artifact
                    self.results.append(artifact)
                    logger.info('[orch.stream] ARTIFACT name=%s', artifact.name)
                    if artifact.name == 'PlannerAgent-result':
                        artifact_data = json_format.MessageToDict(artifact.parts[0].data)
                        if 'trip_info' in artifact_data:
                            self.travel_context = artifact_data['trip_info']
                        current_node_id = start_node_id
                        for idx, task_data in enumerate(artifact_data['tasks']):
                            node = self._add_graph_node(task_id, context_id, task_data['description'], node_id=current_node_id)
                            current_node_id = node.id
                            if idx == 0:
                                should_resume = True
                                start_node_id = node.id
                    else:
                        continue

                if not should_resume:
                    logger.info('[orch.stream] YIELD payload=%s', payload_type)
                    yield (stream_resp, task)

            if not should_resume:
                break

        if self.graph.state == Status.COMPLETED:
            summary = await self._generate_summary()
            self._clear_state()
            logger.info('[orch.stream] COMPLETED summary_len=%d', len(summary or ''))
            yield {'response_type': 'text', 'is_task_complete': True, 'require_user_input': False, 'content': summary}

In [ ]:

app = build_a2a_app(OrchestratorAgent(), 'agent_cards/orchestrator_agent.json')

#config = uvicorn.Config(app=app, host='localhost', port=10101, log_level='info')
#server = uvicorn.Server(config)
#asyncio.ensure_future(server.serve())
#print('Orchestrator Agent running at http://localhost:10101')

if __name__ == "__main__":
    config = uvicorn.Config(app=app, host='localhost', port=10101)
    server = uvicorn.Server(config)
    await server.serve()

INFO:     Started server process [8293]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:10101 (Press CTRL+C to quit)


INFO:     ::1:56678 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     ::1:56679 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     ::1:56679 - "POST / HTTP/1.1" 200 OK


weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d9741-2567-7c4d-a0c4-faed9879d667


INFO:     ::1:56690 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     ::1:56690 - "POST / HTTP/1.1" 200 OK


weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d9741-6aef-7d4d-96aa-18fb32f51e18


...

weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d9747-4371-7a35-954c-2d07dc8f10e0


INFO:     ::1:56725 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     ::1:56725 - "POST / HTTP/1.1" 200 OK


weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d9747-64de-7079-aff5-33b47ce50341


INFO:     ::1:56733 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     ::1:56733 - "POST / HTTP/1.1" 200 OK


weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019d9747-8bae-744d-a994-9a6ddeebbfa1
